# Extract Prodcom data

## Notebook README

**Related publication**

If you utilize any portions of the code, results, or draw inspiration for your projects, please give proper credit by citing the published article below.

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This code extracts relevant products data from the Eurostat PRODCOM database. Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**
 
 - May 04, 2026: Ready for submission
 - May 08, 2026: Retought the `forecast` function, including conversion of units and specific year setting, renamed `get_amount_year_unit_tuple`. Revamped the whole notebooks to permit the analysis of a list of years and regions. Clarified variable names.

**Package versions**

 - See [`environment.yml`](./environment.yml)

**Relevant references**

 - Eurostat (2023) “Statistical Classification of Economic Activities in the European Community, Rev. 2.1 (NACE Rev. 2.1).” Available at: https://showvoc.op.europa.eu/#/datasets/ESTAT_Statistical_Classification_of_Economic_Activities_in_the_European_Community_Rev._2.1._(NACE_2.1).
 - Eurostat (2026) “PRODCOM. Statistics by products. Sold production, exports and imports (ds-059358).” Available at: https://ec.europa.eu/eurostat/databrowser/product/view/ds-059358?category=prom.
 - Eurostat (2025) “Database - Prodcom.” Available at: https://ec.europa.eu/eurostat/web/prodcom/database.
 - Eurostat (2023) European business statistics user’s manual for PRODCOM: 2023 edition. 2023 edition. LU: Publications Office. Available at: https://data.europa.eu/doi/10.2785/39767 (Accessed: May 15, 2024).
 - Eurostat (2025) Quick guide to accessing PRODCOM data in the Eurostat’s Data Browser DS-059358. Available at: https://ec.europa.eu/eurostat/documents/120432/19597181/Quick+guide+on+accessing+PRODCOM+data+DS-059358.pdf.

**Licence**

This work is licenced under the Creative Commons Attribution (CC-BY 4.0) public licence.

## Imports and initialization

In [1]:
import pandas as pd
from numbers import Number
from datetime import datetime
from importlib.metadata import version

In [2]:
print('pandas:', version('pandas'))

pandas: 2.2.3


In [3]:
TODAY_DATE_STR = datetime.now().strftime('%Y%m%d')

In [4]:
OUTPUT_FOLDER = "../Output_data"
INPUT_FOLDER = "../Prodcom_data"
OTHER_DATA_FOLDER = "../Other_data"
SELECTED_YEARS = ['2019', '2022']
SELECTED_REGIONS = ['EU27_2020', 'FR']
SELECTED_INDICATORS = ['PRODQNT', 'EXPQNT', 'IMPQNT']

# NOTE: Relevant NACE codes:
# - 20.14 Other organic basic chemicals (en)
# - 20.16 Plastics in primary forms (en)
# Source : 05-02-2026 - https://showvoc.op.europa.eu/#/datasets/ESTAT_Statistical_Classification_of_Economic_Activities_in_the_European_Community_Rev._2.1._(NACE_2.1)/
SELECTED_NACE_CODES = ['2014', '2016'] 

KG_CONVERSION_FACTORS = {
    '20147400': 0.79, # Undenatured ethyl alcohol of an alcoholic strength by volume >= 80 % (important: excluding alcohol duty) - Source : 04-02-2026 - https://pubchem.ncbi.nlm.nih.gov/compound/Ethanol#section=Density&fullscreen=true
    '20147500': 0.79, # Denatured ethyl alcohol and other denatured spirits; of any strength - Source : 04-02-2026 - https://pubchem.ncbi.nlm.nih.gov/compound/Ethanol#section=Density&fullscreen=true
}

## Import files

In [6]:
# NOTE: Repository link: https://ec.europa.eu/eurostat/databrowser/bulk?lang=en&alphabeticalFilter=D&searchFilter=DS
# NOTE: Took <10s
input_prodcom_df = pd.read_csv(INPUT_FOLDER + '/estat_ds-059358.tsv', sep='\t')
input_prodcom_df

,"freq,reporter,product,indicators\TIME_PERIOD",1995,1996,1997,1998,1999,2000,2001,2002,2003,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,"A,AL,07101010,OWNQNTFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
1,"A,AL,07101010,OWNVALFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
2,"A,AL,07101010,PQNTFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
3,"A,AL,07101010,PVALFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
4,"A,AL,07101010,QNTUNIT",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,KG,KG,KG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171275,"A,XS,399901Z8,OWNQNTFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
2171276,"A,XS,399901Z8,OWNVALFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
2171277,"A,XS,399901Z8,PQNTFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C
2171278,"A,XS,399901Z8,PVALFLAG",:,:,:,:,:,:,:,:,:,...,:,:,:,:,:,:,:,:C,:C,:C


In [7]:
# NOTE: Extracted from Eurostat using Data Browser. Access link (may be deactivated by Eurostat in the future): https://ec.europa.eu/eurostat/databrowser/view/ds-059358__custom_19869966/default/table
prodcom_classification = pd.read_excel(INPUT_FOLDER + '/ds-059358__custom_19869966_spreadsheet.xlsx', sheet_name='Sheet 1', header=9, usecols='A:B', nrows=4788)
prodcom_classification

/home/nicolas/miniconda3/envs/202604-paper-1-supplementary-code/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,PRODUCT (Codes),PRODUCT (Labels)
0,07101000,Iron ores and concentrates (excluding roasted ...
1,07101010,Iron ores and concentrates. Non-agglomerated (...
2,07101020,Iron ores and concentrates. Agglomerated (excl...
3,07291100,Copper ores and concentrates
4,07291200,Nickel ores and concentrates
...,...,...
4783,399900Z7,Kaolin
4784,399900Z8,"Point-of-sale terminals, ATMs and similar mach..."
4785,399900Z9,Tube or pipe fittings of cast iron and cast steel
4786,399901Z0,Bismuth and articles thereof; unwrought hafniu...


In [5]:
# NOTE: Identify reporter regions of Prodcom
# Download link: https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/codelist/ESTAT/geo?format=TSV
geographies_df = pd.read_csv(OTHER_DATA_FOLDER + '/ESTAT_GEO_27.0.tsv', delimiter="\t")
geographies_df

,EUR,Europe
0,EU,"European Union (EU6-1958, EU9-1973, EU10-1981,..."
1,EU_V,European Union (aggregate changing according t...
2,EU_V_NO,European Union (aggregate changing according t...
3,EU27_2020_EFTA,European Union - 27 countries (from 2020) and ...
4,EU27_2020_IS_K,European Union - 27 countries (from 2020) and ...
...,...,...
4285,NAL,Not allocated
4286,NAP,Not applicable
4287,NRP,No response
4288,NSP,Not specified


## Prepare the subset dataframe

In [9]:
# NOTE: Split key single column into multiple columns
index_columns = 'freq,reporter,product,indicators'.split(',')
index_columns

# NOTE: Transforming the comma separated text in cells to list of text in the same cells
# Source : 28-01-2026 - https://stackoverflow.com/questions/79118853/create-new-dataframe-rows-when-column-has-comma-delimited-values#comment139509998_79118853
# 
# NOTE: On converting the list in cells to elements of new columns
# Source : 28-01-2026 - https://stackoverflow.com/questions/35491274/split-a-pandas-column-of-lists-into-multiple-columns#35491399
new_index = pd.DataFrame(input_prodcom_df['freq,reporter,product,indicators\\TIME_PERIOD'].str.split(',').tolist(), index=input_prodcom_df.index, columns=index_columns)
new_index

,freq,reporter,product,indicators
0,A,AL,07101010,OWNQNTFLAG
1,A,AL,07101010,OWNVALFLAG
2,A,AL,07101010,PQNTFLAG
3,A,AL,07101010,PVALFLAG
4,A,AL,07101010,QNTUNIT
...,...,...,...,...
2171275,A,XS,399901Z8,OWNQNTFLAG
2171276,A,XS,399901Z8,OWNVALFLAG
2171277,A,XS,399901Z8,PQNTFLAG
2171278,A,XS,399901Z8,PVALFLAG


In [10]:
# input_prodcom_df_copy = input_prodcom_df
input_prodcom_df_copy = input_prodcom_df.copy()

In [11]:
# NOTE: Generate MultiIndex from DataFrame
# Source : 28-01-2026 - https://pandas.pydata.org/docs/user_guide/advanced.html
input_prodcom_df_copy.index = pd.MultiIndex.from_frame(new_index)

In [12]:
columns = input_prodcom_df_copy.columns
columns

Index(['freq,reporter,product,indicators\TIME_PERIOD', '1995 ', '1996 ',
       '1997 ', '1998 ', '1999 ', '2000 ', '2001 ', '2002 ', '2003 ', '2004 ',
       '2005 ', '2006 ', '2007 ', '2008 ', '2009 ', '2010 ', '2011 ', '2012 ',
       '2013 ', '2014 ', '2015 ', '2016 ', '2017 ', '2018 ', '2019 ', '2020 ',
       '2021 ', '2022 ', '2023 ', '2024 '],
      dtype='object')

In [13]:
# NOTE: Remove space in column labels
input_prodcom_df_copy.columns = columns.map(lambda x: x.strip())

# NOTE: Remove redundant column
input_prodcom_df_copy.drop('freq,reporter,product,indicators\\TIME_PERIOD', axis=1, inplace=True)
input_prodcom_df_copy

1995 1996 1997 1998 1999 2000 2001 2002  \
freq reporter product  indicators                                           
A    AL       07101010 OWNQNTFLAG   :    :    :    :    :    :    :    :    
                       OWNVALFLAG   :    :    :    :    :    :    :    :    
                       PQNTFLAG     :    :    :    :    :    :    :    :    
                       PVALFLAG     :    :    :    :    :    :    :    :    
                       QNTUNIT      :    :    :    :    :    :    :    :    
...                                ...  ...  ...  ...  ...  ...  ...  ...   
     XS       399901Z8 OWNQNTFLAG   :    :    :    :    :    :    :    :    
                       OWNVALFLAG   :    :    :    :    :    :    :    :    
                       PQNTFLAG     :    :    :    :    :    :    :    :    
                       PVALFLAG     :    :    :    :    :    :    :    :    
                       QNTUNIT      :    :    :    :    :    :    :    :    

                                  2003 2004  ... 2015 2016 2017 2018 2019  \
freq reporter product  indicators            ...                            
A    AL       07101010 OWNQNTFLAG   :    :   ...   :    :    :    :    :    
                       OWNVALFLAG   :    :   ...   :    :    :    :    :    
                       PQNTFLAG     :    :   ...   :    :    :    :    :    
                       PVALFLAG     :    :   ...   :    :    :    :    :    
                       QNTUNIT      :    :   ...   :    :    :    :    :    
...                                ...  ...  ...  ...  ...  ...  ...  ...   
     XS       399901Z8 OWNQNTFLAG   :    :   ...   :    :    :    :    :    
                       OWNVALFLAG   :    :   ...   :    :    :    :    :    
                       PQNTFLAG     :    :   ...   :    :    :    :    :    
                       PVALFLAG     :    :   ...   :    :    :    :    :    
                       QNTUNIT      :    :   ...   :    :    :    :    :    

                                  2020 2021 2022 2023 2024  
freq reporter product  indicators                           
A    AL       07101010 OWNQNTFLAG   :    :    :C   :C   :C  
                       OWNVALFLAG   :    :    :C   :C   :C  
                       PQNTFLAG     :    :    :C   :C   :C  
                       PVALFLAG     :    :    :C   :C   :C  
                       QNTUNIT      :    :    KG   KG   KG  
...                                ...  ...  ...  ...  ...  
     XS       399901Z8 OWNQNTFLAG   :    :    :C   :C   :C  
                       OWNVALFLAG   :    :    :C   :C   :C  
                       PQNTFLAG     :    :    :C   :C   :C  
                       PVALFLAG     :    :    :C   :C   :C  
                       QNTUNIT      :    :   PST  PST  PST  

[2171280 rows x 30 columns]

In [14]:
# NOTE: Filter the DataFrame on
# - 'freq' = 'A'
# - 'reporter' = 'EU27_2020'
# - 'product' code starts with either '2014' or '2016'
# - 'indicators' = 'PRODQNT', 'EXPQNT' or 'IMPQNT'

filtered_prodcom_df = input_prodcom_df_copy.loc[
    (input_prodcom_df_copy.index.get_level_values('freq') == 'A')
    & input_prodcom_df_copy.index.get_level_values('reporter').isin(SELECTED_REGIONS)
    & input_prodcom_df_copy.index.get_level_values('product').str.startswith(tuple(SELECTED_NACE_CODES))
    # & new_df.index.get_level_values('product').str.startswith(tuple(cpa_codes_filter.to_list())) # CPA codes obtained from EXIOBASE matching
    & input_prodcom_df_copy.index.get_level_values('indicators').isin(SELECTED_INDICATORS)
    # & (new_df.index.get_level_values('indicators').str.endswith('QNT') | new_df.index.get_level_values('indicators').str.endswith('VAL'))
]
filtered_prodcom_df

1995     1996     1997     1998  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20141130 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
...                                     ...      ...      ...      ...   
     FR        20165965 IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20165970 EXPQNT            0        0        0        0   
                        IMPQNT      7891100  6412100  7266600  7096700   
                        PRODQNT          :        :        :        :    

                                       1999     2000     2001     2002  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20141130 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
...                                     ...      ...      ...      ...   
     FR        20165965 IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20165970 EXPQNT            0        0        0        0   
                        IMPQNT      7568600  8633100  6196100  8004000   
                        PRODQNT          :        :        :        :    

                                          2003        2004  ...        2015  \
freq reporter  product  indicators                          ...               
A    EU27_2020 20141120 EXPQNT        26356100    41433500  ...    49195700   
                        IMPQNT       890151000   587529700  ...  1729281200   
                        PRODQNT             :           :   ...  1167702384   
               20141130 EXPQNT       183744600   117247400  ...   178548900   
                        IMPQNT      1567331300  1289242200  ...  1128849700   
...                                        ...         ...  ...         ...   
     FR        20165965 IMPQNT              :           :   ...          :    
                        PRODQNT             :           :   ...          :    
               20165970 EXPQNT              :            0  ...           0   
                        IMPQNT        11415900     9411800  ...     8552900   
                        PRODQNT             :           :   ...          :    

                                          2016        2017        2018  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT        60905400    80848600    69913206   
                        IMPQNT      1905384600  1914049400  2169914471   
                        PRODQNT     1056243997   884248263   901285653   
               20141130 EXPQNT       512366300   389225700   602736255   
                        IMPQNT      1249475500  1326164900  1376816315   
...                                        ...         ...         ...   
     FR        20165965 IMPQNT              :           :           :    
                        PRODQNT             :           :           :    
               20165970 EXPQNT               0           0           0   
                        IMPQNT         9805800     9706000     8947500   
                        PRODQNT             :           :           :    

                                          2019        2020        2021  \
freq reporter  product  indicators              

In [38]:
prodcom_units_df = input_prodcom_df_copy.loc[
    (input_prodcom_df_copy.index.get_level_values('freq') == 'A')
    & input_prodcom_df_copy.index.get_level_values('reporter').isin(SELECTED_REGIONS)
    & input_prodcom_df_copy.index.get_level_values('product').str.startswith(tuple(SELECTED_NACE_CODES))
    & input_prodcom_df_copy.index.get_level_values('indicators').str.endswith('UNIT')
]
prodcom_units_df

1995 1996 1997 1998 1999 2000 2001 2002  \
freq reporter  product  indicators                                           
A    EU27_2020 20141120 QNTUNIT      :    :    :    :    :    :    :    :    
               20141130 QNTUNIT      :    :    :    :    :    :    :    :    
               20141140 QNTUNIT      :    :    :    :    :    :    :    :    
               20141150 QNTUNIT      :    :    :    :    :    :    :    :    
               20141160 QNTUNIT      :    :    :    :    :    :    :    :    
...                                 ...  ...  ...  ...  ...  ...  ...  ...   
     FR        20165950 QNTUNIT      :    :    :    :    :    :    :    :    
               20165955 QNTUNIT      :    :    :    :    :    :    :    :    
               20165960 QNTUNIT      KG   KG   KG   KG   KG   KG   KG   KG   
               20165965 QNTUNIT      :    :    :    :    :    :    :    :    
               20165970 QNTUNIT      KG   KG   KG   KG   KG   KG   KG   KG   

                                   2003 2004  ... 2015 2016 2017 2018 2019  \
freq reporter  product  indicators            ...                            
A    EU27_2020 20141120 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   KG   
               20141130 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   KG   
               20141140 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   KG   
               20141150 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   KG   
               20141160 QNTUNIT      :    :   ...   KG   KG   KG   KG   KG   
...                                 ...  ...  ...  ...  ...  ...  ...  ...   
     FR        20165950 QNTUNIT      :    :   ...   :    :    :    :    KG   
               20165955 QNTUNIT      :    :   ...   :    :    :    :    KG   
               20165960 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   :    
               20165965 QNTUNIT      :    :   ...   :    :    :    :    KG   
               20165970 QNTUNIT      KG   KG  ...   KG   KG   KG   KG   KG   

                                   2020 2021 2022 2023 2024  
freq reporter  product  indicators                           
A    EU27_2020 20141120 QNTUNIT      KG   KG   KG   KG   KG  
               20141130 QNTUNIT      KG   KG   KG   KG   KG  
               20141140 QNTUNIT      KG   KG   KG   KG   KG  
               20141150 QNTUNIT      KG   KG   KG   KG   KG  
               20141160 QNTUNIT      KG   KG   KG   KG   KG  
...                                 ...  ...  ...  ...  ...  
     FR        20165950 QNTUNIT      KG   KG   KG   KG   KG  
               20165955 QNTUNIT      KG   KG   KG   KG   KG  
               20165960 QNTUNIT      :    :    :    :    :   
               20165965 QNTUNIT      KG   KG   KG   KG   KG  
               20165970 QNTUNIT      KG   KG   KG   KG   KG  

[470 rows x 30 columns]

## Extract values

In [16]:
product_codes = filtered_prodcom_df.index.get_level_values('product').unique()
product_codes

Index(['20141120', '20141130', '20141140', '20141150', '20141160', '20141165',
       '20141167', '20141190', '20141213', '20141215',
       ...
       '20165670', '20165700', '20165920', '20165940', '20165945', '20165950',
       '20165955', '20165960', '20165965', '20165970'],
      dtype='object', name='product', length=235)

In [100]:
# NOTE: Estimate quantity at a certain date
# Source : 29-01-2026 - https://codepointtech.com/how-to-find-first-row-that-meets-criteria-in-pandas-effectively/

def get_amount_year_unit_tuple(row_series, year:str = None):

    row_index_keys = row_series.name

    # Take the most recent year where it is available
    if year is None:
        criteria = (row_series.str.strip() != ':') & row_series.str.isnumeric()

        # NOTE: Take the lastest available data index
        last_match_index = criteria.nlargest(1, keep='last').index
        year = last_match_index[0]
    # Get data for the specific `year`
    else:
        last_match_index = pd.Index([year])

    matching_row = row_series.loc[last_match_index]

    amount_cell = matching_row.iloc[0]
    # In theory, could only happen in some cases when `year` parameter is specifically defined
    if amount_cell.strip() != ':' and amount_cell.isnumeric():
        amount = int(amount_cell)
    else:
        amount = amount_cell.strip()

    # NOTE: Retrieve unit from the relevant row
    unit = prodcom_units_df.loc[
                    (prodcom_units_df.index.get_level_values('reporter').values == row_index_keys[1]) &
                    (prodcom_units_df.index.get_level_values('product').values == row_index_keys[2])
                ].iloc[0][last_match_index].iloc[0]
    # NOTE: Convert to KG
    if row_index_keys[2] in KG_CONVERSION_FACTORS and isinstance(amount, Number) and unit != 'KG':
                amount = amount * KG_CONVERSION_FACTORS[row_index_keys[2]]
                unit = 'KG'
        

    # NOTE: Retrive product name
    # description = ''
    # description_row = prodcom_classification.loc[lambda x : x['PRODUCT (Codes)'] == row_name[2]]
    # if len(description_row):
    #     description = description_row.loc(axis=1)['PRODUCT (Labels)'].iloc[0]

    # first_match_index = criteria.idxmax()
    # Retrieve the row using .loc
    # first_matching_row = series.loc[first_match_index]

    # return series.loc[lambda x: x.str.strip() != ':'].tail(1).astype('int').values
    
    return (amount, unit)

In [102]:
for year in SELECTED_YEARS:
    latest_df = filtered_prodcom_df.apply(get_amount_year_unit_tuple, axis=1, result_type='expand', year=year)
    latest_df.columns=[f"amount - {year}", f"unit - {year}"]

    filtered_prodcom_df = filtered_prodcom_df.join(latest_df)

filtered_prodcom_df

1995     1996     1997     1998  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20141130 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
...                                     ...      ...      ...      ...   
     FR        20165965 IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20165970 EXPQNT            0        0        0        0   
                        IMPQNT      7891100  6412100  7266600  7096700   
                        PRODQNT          :        :        :        :    

                                       1999     2000     2001     2002  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20141130 EXPQNT           :        :        :        :    
                        IMPQNT           :        :        :        :    
...                                     ...      ...      ...      ...   
     FR        20165965 IMPQNT           :        :        :        :    
                        PRODQNT          :        :        :        :    
               20165970 EXPQNT            0        0        0        0   
                        IMPQNT      7568600  8633100  6196100  8004000   
                        PRODQNT          :        :        :        :    

                                          2003        2004  ...        2019  \
freq reporter  product  indicators                          ...               
A    EU27_2020 20141120 EXPQNT        26356100    41433500  ...    94938259   
                        IMPQNT       890151000   587529700  ...  1993784447   
                        PRODQNT             :           :   ...   861284640   
               20141130 EXPQNT       183744600   117247400  ...   280758738   
                        IMPQNT      1567331300  1289242200  ...  1418579590   
...                                        ...         ...  ...         ...   
     FR        20165965 IMPQNT              :           :   ...    12281700   
                        PRODQNT             :           :   ...     8949049   
               20165970 EXPQNT              :            0  ...           0   
                        IMPQNT        11415900     9411800  ...    10387800   
                        PRODQNT             :           :   ...           0   

                                          2020        2021        2022  \
freq reporter  product  indicators                                       
A    EU27_2020 20141120 EXPQNT        77677977    81800176    74915115   
                        IMPQNT      1481110094  1698091954  1792881362   
                        PRODQNT      787049864  1040000000   980000000   
               20141130 EXPQNT       419917128   208194049   130155532   
                        IMPQNT      1297094111   889851883  1250333999   
...                                        ...         ...         ...   
     FR        20165965 IMPQNT        13858149    12169468    10409845   
                        PRODQNT        7407584          :           :    
               20165970 EXPQNT               0    16797806    17810698   
                        IMPQNT         7572399     8421500    12673156   
                        PRODQNT             :           :           :    

                                          2023        2024 amount - 2019  \
freq reporter  product  indicators            

In [ ]:
# NOTE: Check that the unit conversion went fine
filtered_prodcom_df.loc[filtered_prodcom_df.index.get_level_values('product').isin(list(KG_CONVERSION_FACTORS.keys()))]

1995       1996       1997  \
freq reporter  product  indicators                                    
A    EU27_2020 20147400 EXPQNT             :          :          :    
                        IMPQNT             :          :          :    
                        PRODQNT            :          :          :    
               20147500 EXPQNT             :          :          :    
                        IMPQNT             :          :          :    
                        PRODQNT            :          :          :    
     FR        20147400 EXPQNT      366746247  318983050  266920638   
                        IMPQNT      156677892  171259620  125092522   
                        PRODQNT     612470605  517612878  518088439   
               20147500 EXPQNT       46260295   50125642   56140046   
                        IMPQNT         979957    2225147    1319941   
                        PRODQNT      57663617   16300613  156566858   

                                         1998       1999       2000  \
freq reporter  product  indicators                                    
A    EU27_2020 20147400 EXPQNT             :          :          :    
                        IMPQNT             :          :          :    
                        PRODQNT            :          :          :    
               20147500 EXPQNT             :          :          :    
                        IMPQNT             :          :          :    
                        PRODQNT            :          :          :    
     FR        20147400 EXPQNT      294599372  263656066  287736638   
                        IMPQNT       87627220  112637389   92668967   
                        PRODQNT     603076206  720595952  795166499   
               20147500 EXPQNT       68615554   71854413   76088986   
                        IMPQNT         947659    2351703    7144422   
                        PRODQNT            :    24672360   28512173   

                                         2001       2002        2003  \
freq reporter  product  indicators                                     
A    EU27_2020 20147400 EXPQNT             :          :    163125632   
                        IMPQNT             :          :    446554589   
                        PRODQNT            :          :   1643419033   
               20147500 EXPQNT             :          :      5825979   
                        IMPQNT             :          :     44038424   
                        PRODQNT            :          :    588481726   
     FR        20147400 EXPQNT      249859986  251802180   266751157   
                        IMPQNT      108743025  106958510   102904281   
                        PRODQNT     783968081  719412163   651839189   
               20147500 EXPQNT       73844228   84913766    97805713   
                        IMPQNT        6442837    3533291     4943807   
                        PRODQNT      44269210  104355291   133200995   

                                          2004  ...        2019        2020  \
freq reporter  product  indicators              ...                           
A    EU27_2020 20147400 EXPQNT       176267028  ...   732921818   570589295   
                        IMPQNT       446989111  ...   970013659  1027590979   
                        PRODQNT     1586790860  ...  4872416900  4821375676   
               20147500 EXPQNT         9384404  ...    83954545   118282505   
                        IMPQNT        62276090  ...   230363051   562562201   
                        PRODQNT      692845488  ...  1143715201  1211043281   
     FR        20147400 EXPQNT       277308673  ...   767316131   719853791   
                        IMPQNT       105048838  ...   449256661   523146700   
                        PRODQNT      628835448  ...  1436132871  1342461759   
               20147500 EXPQNT        97561189  ...   120256006   151425719   
                        IMPQNT        13140973  ...    11028266    14522216   
               

In [ ]:
# NOTE: Temporary save
# filtered_prodcom_with_latest_df.to_pickle(OUTPUT_PATH + 'filtered_prodcom_with_latest_df.pkl')

In [ ]:
# NOTE: Load backed-up temporary data
# filtered_prodcom_with_latest_df = pd.read_pickle(OUTPUT_PATH + 'filtered_prodcom_with_latest_df.pkl')

## Transform result

In [115]:
# NOTE: Generate the new MultiIndex
filtered_prodcom_grouped_index = filtered_prodcom_df.index
filtered_prodcom_grouped_index = filtered_prodcom_grouped_index.droplevel(['reporter', 'indicators'])
filtered_prodcom_grouped_index = filtered_prodcom_grouped_index.drop_duplicates()
filtered_prodcom_grouped_index

MultiIndex([('A', '20141120'),
            ('A', '20141130'),
            ('A', '20141140'),
            ('A', '20141150'),
            ('A', '20141160'),
            ('A', '20141165'),
            ('A', '20141167'),
            ('A', '20141190'),
            ('A', '20141213'),
            ('A', '20141215'),
            ...
            ('A', '20165670'),
            ('A', '20165700'),
            ('A', '20165920'),
            ('A', '20165940'),
            ('A', '20165945'),
            ('A', '20165950'),
            ('A', '20165955'),
            ('A', '20165960'),
            ('A', '20165965'),
            ('A', '20165970')],
           names=['freq', 'product'], length=235)

In [116]:
# NOTE: Create a new DataFrame using the new MultiIndex
filtered_prodcom_grouped_df = pd.DataFrame(index=filtered_prodcom_grouped_index)
filtered_prodcom_grouped_df

Empty DataFrame
Columns: []
Index: [(A, 20141120), (A, 20141130), (A, 20141140), (A, 20141150), (A, 20141160), (A, 20141165), (A, 20141167), (A, 20141190), (A, 20141213), (A, 20141215), (A, 20141223), (A, 20141225), (A, 20141243), (A, 20141245), (A, 20141247), (A, 20141250), (A, 20141260), (A, 20141270), (A, 20141290), (A, 20141313), (A, 20141315), (A, 20141323), (A, 20141325), (A, 20141353), (A, 20141357), (A, 20141371), (A, 20141374), (A, 20141379), (A, 20141450), (A, 20141470), (A, 20141490), (A, 20141910), (A, 20141930), (A, 20141950), (A, 20141970), (A, 20142100), (A, 20142210), (A, 20142220), (A, 20142230), (A, 20142240), (A, 20142263), (A, 20142265), (A, 20142270), (A, 20142310), (A, 20142320), (A, 20142333), (A, 20142337), (A, 20142338), (A, 20142339), (A, 20142350), (A, 20142360), (A, 20142373), (A, 20142375), (A, 20142410), (A, 20142433), (A, 20142439), (A, 20142450), (A, 20143120), (A, 20143130), (A, 20143150), (A, 20143195), (A, 20143197), (A, 20143215), (A, 20143219), (A, 20143220), (A, 20143230), (A, 20143235), (A, 20143240), (A, 20143250), (A, 20143271), (A, 20143277), (A, 20143278), (A, 20143280), (A, 20143310), (A, 20143320), (A, 20143330), (A, 20143340), (A, 20143350), (A, 20143363), (A, 20143365), (A, 20143367), (A, 20143370), (A, 20143381), (A, 20143382), (A, 20143383), (A, 20143385), (A, 20143387), (A, 20143410), (A, 20143420), (A, 20143430), (A, 20143440), (A, 20143473), (A, 20143475), (A, 201434Z1), (A, 20144113), (A, 20144119), (A, 20144123), (A, 20144129), (A, 20144130), (A, 20144151), ...]

[235 rows x 0 columns]

In [117]:
# NOTE: Get the label name of a Prodcom code
def get_label(code):
    description = ''
    description_row = prodcom_classification.loc[lambda x : x['PRODUCT (Codes)'] == code]
    if len(description_row):
        description = description_row.loc(axis=1)['PRODUCT (Labels)'].iloc[0]
    return description

In [118]:
# NOTE: Insert Prodcom product label text column
filtered_prodcom_grouped_df['product label'] = ''
# Fill with content
filtered_prodcom_grouped_df['product label'] = filtered_prodcom_grouped_df.apply(lambda row: get_label(row.name[1]), axis=1)
filtered_prodcom_grouped_df

product label
freq product                                                    
A    20141120                     Saturated acyclic hydrocarbons
     20141130                                           Ethylene
     20141140                                Propene (propylene)
     20141150              Butene (butylene) and isomers thereof
     20141160                        Buta-1,3-diene and isoprene
...                                                          ...
     20165950  Cellulose and its chemical derivatives, in pri...
     20165955  Natural polymers and modified natural polymers...
     20165960  Natural and modified natural polymers, in prim...
     20165965  Natural polymers and modified natural polymers...
     20165970  Ion-exchangers based on synthetic or natural p...

[235 rows x 1 columns]

In [119]:
# NOTE: Return the value for a specific product key ('row_name') and for the selected indicator
def get_column_value(row, region, indicator, col, year):

    row_name = row.name

    column_name = f"{col} - {year}"

    row = filtered_prodcom_df.loc[
        (filtered_prodcom_df.index.get_level_values('freq').values == row_name[0])
        & (filtered_prodcom_df.index.get_level_values('reporter').values == region)
        & (filtered_prodcom_df.index.get_level_values('product').values == row_name[1])
        & (filtered_prodcom_df.index.get_level_values('indicators').values == indicator),
        column_name
    ]
    if len(row):
        return row.iloc[0]
    else:
        return 'N/A'

In [ ]:
# NOTE: Generate pairs of columns ('amount', 'unit') for each selected indicator (e.g., 'PRODQNT', 'EXPQNT', 'IMPQNT') and regions (e.g., 'EU27_2020', 'FR')

for region in SELECTED_REGIONS:
    for year in SELECTED_YEARS:
        for indicator in SELECTED_INDICATORS:
            for col in ["amount", "unit"]:
                cell_value = filtered_prodcom_grouped_df.apply(get_column_value, axis=1, region=region, indicator=indicator, col=col, year=year)
                filtered_prodcom_grouped_df[f"{indicator} - {region} - {year} - {col}"] = cell_value

        # Sum Imports and Exports (not continued)
        # filtered_prodcom_grouped_df[f"PRODQNT IMPQNT  - {region} - {year} - {col}"]


filtered_prodcom_grouped_df

product label  \
freq product                                                       
A    20141120                     Saturated acyclic hydrocarbons   
     20141130                                           Ethylene   
     20141140                                Propene (propylene)   
     20141150              Butene (butylene) and isomers thereof   
     20141160                        Buta-1,3-diene and isoprene   
...                                                          ...   
     20165950  Cellulose and its chemical derivatives, in pri...   
     20165955  Natural polymers and modified natural polymers...   
     20165960  Natural and modified natural polymers, in prim...   
     20165965  Natural polymers and modified natural polymers...   
     20165970  Ion-exchangers based on synthetic or natural p...   

              PRODQNT - EU27_2020 - 2019 - amount  \
freq product                                        
A    20141120                           861284640   
     20141130                          9817689027   
     20141140                          9792446126   
     20141150                          1674613555   
     20141160                          2742866596   
...                                           ...   
     20165950                           342232085   
     20165955                           134000000   
     20165960                                   :   
     20165965                           100000000   
     20165970                           142099386   

              PRODQNT - EU27_2020 - 2019 - unit  \
freq product                                      
A    20141120                                KG   
     20141130                                KG   
     20141140                                KG   
     20141150                                KG   
     20141160                                KG   
...                                         ...   
     20165950                                KG   
     20165955                                KG   
     20165960                                :    
     20165965                                KG   
     20165970                                KG   

              EXPQNT - EU27_2020 - 2019 - amount  \
freq product                                       
A    20141120                           94938259   
     20141130                          280758738   
     20141140                          124453445   
     20141150                           95308472   
     20141160                          298433161   
...                                          ...   
     20165950                          215327703   
     20165955                            5423076   
     20165960                                  :   
     20165965                           21277604   
     20165970                           17426520   

              EXPQNT - EU27_2020 - 2019 - unit  \
freq product                                     
A    20141120                               KG   
     20141130                               KG   
     20141140                               KG   
     20141150                               KG   
     20141160                               KG   
...                                        ...   
     20165950                               KG   
     20165955                               KG   
     20165960                               :    
     20165965                               KG   
     20165970                               KG   

              IMPQNT - EU27_2020 - 2019 - amount  \
freq product                                       
A    20141120                         1993784447   
     20141130                         1418579590   
     20141140                          845569901   
     20141150                              66119   
     20141160                           27881078   
...                                          ...   
     20165950                          115215823   
     20165955   

In [ ]:
# NOTE: Temporary save
# filtered_prodcom_grouped_df.to_pickle(OUTPUT_PATH + 'filtered_prodcom_grouped_df.pkl')

In [121]:
# NOTE: Export to CSV
filtered_prodcom_grouped_df.to_csv(f"{OUTPUT_FOLDER}/{TODAY_DATE_STR} - filtered_prodcom_grouped_{"-".join(SELECTED_REGIONS)}_{"-".join(SELECTED_YEARS)}.csv")